# HyWindSea SANTANDER — Case selection

Two-stage selection of the wind conditions that drive the metamodel:

1. **WRF days** — which days (out of 1982–2023) to simulate at high resolution. Chosen with PCA + MDA on ERA5 *daily* wind maps. **All 365 selected days are simulated with WRF.**
2. **SWAN cases** — which hours of those 365 days to simulate with SWAN. Chosen with PCA + MDA on the 8 760 ERA5 *hourly* wind fields. **All 365 × 24 hours are ranked by MDA; this notebook shows the 10 most dissimilar as a worked example. The production RBF (`03_build_rbf`) uses 500 cases.**

The PCA fitted in stage 2 is the metamodel predictor reused by `03`, `04` and `05`.

**Reads** `inputs/daily_era5_1982-2023.nc`, `inputs/wind_era5_selected_days.nc`  
**Writes** `outputs/SANTANDER/times/times_wrf_santander.txt`  
         `outputs/SANTANDER/pca/pca_ERA5_SANTANDER.pkl`  
         `outputs/SANTANDER/times/times{NUM_TIMES}.txt`

In [1]:
import os

import numpy as np
import pandas as pd
import xarray as xr
from bluemath_tk.datamining.mda import MDA
from bluemath_tk.datamining.pca import PCA

from utils.operations import max_diss

In [2]:
SITE      = "SANTANDER"

In [3]:
                

PCA_DIR   = f"outputs/{SITE}/pca"
TIMES_DIR = f"outputs/{SITE}/times"
os.makedirs(PCA_DIR,   exist_ok=True)
os.makedirs(TIMES_DIR, exist_ok=True)

---
## Phase 1 — WRF day selection

### ERA5 data

ERA5 wind components (u10, v10) were downloaded from the **Copernicus Climate Data Store OPeNDAP** endpoint (`https://cds.climate.copernicus.eu/`).  Access requires a CDS API token stored in `~/.cdsapirc`.

The request retrieved:
- Variables: `10m_u_component_of_wind`, `10m_v_component_of_wind`
- Period: 1982-01-01 – 2023-12-31
- Spatial domain: the ERA5 nodes enclosing the Santander bay (4 lat × 5 lon)
- Temporal resolution: **hourly** → aggregated to **daily means** before storage

The resulting file `inputs/daily_era5_1982-2023.nc` contains one wind map per calendar day (~15 000 maps).

In [5]:
# daily = xr.open_dataset("https://geoocean.unican.es/thredds/dap4/rawData/GEOOCEAN/BlueMath/methods/hybrid_downscaling/metamodels/HyWindSea/inputs/daily_era5_1982-2023.nc#dap4") 
daily = xr.open_dataset(r"D:\03_CLAUDE\hywindsea\repositorio\HyWindSea_SANTANDER\new_repo\inputs\daily_era5_1982-2023.nc") 

daily = daily.sel(time=slice(*("1982", "2023")))  # period covered by ERA5 download


print(f"{daily.sizes['time']:,} days  ×  "
      f"{daily.sizes['lat'] * daily.sizes['lon']} ERA5 nodes")

15,340 days  ×  20 ERA5 nodes


### PCA

Each daily map is flattened to a vector of 40 numbers (4 lat × 5 lon × 2 components). Five PCs retain 98.6 % of the variance and make the downstream MDA tractable.

In [6]:
pca_daily = PCA(n_components=5) # PCs for daily selection (98.6 % variance)

pca_daily.fit_transform(
    data=daily[["u10", "v10"]],
    vars_to_stack=["u10", "v10"],
    coords_to_stack=["lat", "lon"],
    pca_dim_for_rows="time",
    scale_data=True,
)

days = pd.DatetimeIndex(pca_daily.pcs_df.index).normalize()

print(f"{pca_daily.pcs_df.shape[1]} PCs  ·  "
      f"{pca_daily.cumulative_explained_variance_ratio[-1]:.1%} variance retained")

2026-09-16 13:51:04,370 - PCA - WARNING - Using 20 out of 20 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-09-16 13:51:04,373 - PCA - WARNING - Using 20 out of 20 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method


5 PCs  ·  98.6% variance retained


### MDA — select WRF days

Maximum Dissimilarity Algorithm (MDA) iteratively picks the day furthest from all previously selected days in PC space. The result covers the extremes of the wind climate rather than the crowded mean — exactly what a training set needs.

**All 365 days selected here are simulated with WRF** at high resolution (~2.5 km, hourly output), providing the wind forcing for the SWAN runs in the next stage.

In [7]:
order     = max_diss(pca_daily.pcs_df, num_centers=365) # WRF days to simulate
selection = days[order]

print(f"{len(selection)} days selected  "
      f"({selection.min():%Y-%m-%d} → {selection.max():%Y-%m-%d})")
print(f"First five: {', '.join(f'{d:%Y-%m-%d}' for d in selection[:5])} …")

np.savetxt(f"{TIMES_DIR}/times_wrf_santander.txt",
           selection.strftime("%Y-%m-%d").values, fmt="%s")
print(f"\nSaved → {TIMES_DIR}/times_wrf_santander.txt")

365 days selected  (1982-03-07 → 2023-11-04)
First five: 1987-10-15, 2001-11-14, 2018-01-16, 2011-03-16, 1994-08-10 …

Saved → outputs/SANTANDER/times/times_wrf_santander.txt


---
## Phase 2 — SWAN case selection

The 365 WRF days expand to **365 × 24 = 8 760 hourly wind fields**. Running SWAN for all 8 760 hours is not affordable, so MDA is applied again — this time in the PC space of the *hourly* fields — to rank hours by dissimilarity.

This notebook shows the **10 most dissimilar cases** as a worked example to illustrate the full pipeline. The production metamodel (`03_build_rbf` onwards) uses **500 cases**, which the K-Fold analysis confirms is well past the point of diminishing returns.

### Hourly ERA5 for the selected days

Ideally this step would select the 365 WRF days directly from the full ERA5 download and filter to `times_wrf_santander.txt`. To save disk space, the repository ships a pre-filtered file `inputs/wind_era5_full.nc` that already contains only the 8 760 hours corresponding to the selected days — so no additional filtering is needed here.

In [ ]:
# NOTE: in a full setup you would load the complete ERA5 download and filter:
#   era5 = xr.open_dataset("inputs/wind_era5_FULL_1982-2023.nc").sel(time=hours)
# Here wind_era5_full.nc is already pre-filtered to the 365 WRF days (8 760 h)
# to keep the repository lightweight.
wrf_days = pd.DatetimeIndex(
    np.loadtxt(f"{TIMES_DIR}/times_wrf_santander.txt", dtype=str)
)
hours = pd.DatetimeIndex(
    np.concatenate(
        [(day + pd.to_timedelta(np.arange(24), unit="h")).values for day in wrf_days]
    )
)
era5 = xr.open_dataset("https://geoocean.unican.es/thredds/dap4/rawData/GEOOCEAN/BlueMath/methods/hybrid_downscaling/metamodels/HyWindSea/inputs/wind_era5_selected_days.nc#dap4").sel(time=hours)
print(f"{len(wrf_days)} days × 24 h = {era5.sizes['time']:,} hourly wind fields")
era5

365 days × 24 h = 8,760 hourly wind fields


### PCA (metamodel predictor)

This PCA is not only a selection step: the fitted model is saved and reused in `03_build_rbf`, `04_validation` and `05_reconstruction` as the predictor that maps any new ERA5 wind field to a set of PCs.

In [17]:
pca = PCA(n_components=0.995)

pca.fit_transform(
    data=era5,
    vars_to_stack=["u10", "v10"],
    coords_to_stack=["lat", "lon"],
    pca_dim_for_rows="time",
)

print(f"{pca.pca.n_components_} PCs  ·  "
      f"{pca.cumulative_explained_variance_ratio[-1]:.1%} variance retained")

pca.save_model(f"{PCA_DIR}/pca_ERA5_SANTANDER.pkl")
print(f"Saved → {PCA_DIR}/pca_ERA5_SANTANDER.pkl")

2026-09-15 10:13:12,000 - PCA - WARNING - Using 20 out of 20 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-09-15 10:13:12,001 - PCA - WARNING - Using 20 out of 20 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-09-15 10:13:12,014 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!


11 PCs  ·  99.6% variance retained
Saved → outputs/SANTANDER/pca/pca_ERA5_SANTANDER.pkl


### MDA — select SWAN cases

In [19]:
NUM_TIMES = 10                 # SWAN cases (demonstration; production uses 500)


In [18]:
mda = MDA(num_centers=len(pca.pcs_df))
mda.fit(data=pca.pcs_df, directional_variables=[])

times_sel = pca.pcs_df.iloc[mda.centroid_real_indices][:NUM_TIMES]

print(f"{NUM_TIMES} most dissimilar cases selected:")
print(times_sel.index.strftime("%Y-%m-%d %H:%M").tolist())

np.savetxt(
    f"{TIMES_DIR}/times{NUM_TIMES}.txt",
    times_sel.index.values.astype("datetime64[s]").astype(str),
    fmt="%s",
)
print(f"\nSaved → {TIMES_DIR}/times{NUM_TIMES}.txt")

10 most dissimilar cases selected:
['1984-10-04 09:00', '1993-02-27 04:00', '1999-12-27 18:00', '2008-12-27 14:00', '1998-04-03 13:00', '2019-06-27 22:00', '2009-01-24 02:00', '1982-11-06 21:00', '2019-02-02 04:00', '2006-11-24 16:00']

Saved → outputs/SANTANDER/times/times10.txt
